# Example 8 — Arbitrary multiple background categories

Signal + three independent background categories for B+ -> K+ pi+ pi-. The signal fraction is the global reference.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
from dalitzplotfitter import (BackgroundCategory, DecayChannel, DecayModel, Minimizer, NonResonant, Parameter, PhaseSpaceSample, RealImag, Resonance, MultiBackgroundNLL, enable_x64, weighted_resample)
from dalitzplotfitter.background import FunctionalBackground
enable_x64()
channel=DecayChannel('B+',('K+','pi+','pi-'))
components=[Resonance('Kstar892',(0,2),RealImag(1,0),mass=0.8958,width=0.0474,spin=1),Resonance('rho770',(1,2),RealImag(0.65,0.10),mass=0.7753,width=0.1491,spin=1),NonResonant(RealImag(-0.5,0.1))]
model=DecayModel(channel,components,normalization_method='square-dalitz',normalization_resolution=300,normalization_pair=(0,2))


## 1. Define three background shapes


In [ ]:
mK,mpi,_=channel.daughter_masses; mB=channel.parent_mass
s13_min,s13_max=(mK+mpi)**2,(mB-mpi)**2
s23_min,s23_max=(2*mpi)**2,(mB-mK)**2
def scaled(d,k,lo,hi): return jnp.clip((d[k]-lo)/(hi-lo),0,1)
comb=FunctionalBackground(lambda d:0.4+1.1*scaled(d,'s13',s13_min,s13_max))
partial=FunctionalBackground(lambda d:0.25+2.0*(1-scaled(d,'s23',s23_min,s23_max))**2)
misid=FunctionalBackground(lambda d:0.3+1.8*jnp.exp(-0.5*((scaled(d,'s13',s13_min,s13_max)-0.38)/0.12)**2))


## 2. Generate pseudo-data: 72% signal, then 50/30/20% within background


In [ ]:
pool=model.generate_phase_space(160000,seed=8001); norm=model.normalization_sample
N=30000; fs_true=0.72; wc_true=0.50; wp_true=0.30
ns=int(round(N*fs_true)); nb=N-ns; nc=int(round(nb*wc_true)); npart=int(round(nb*wp_true)); nm=nb-nc-npart
weights=[pool.weights*model.intensity(pool.as_dict()),pool.weights*comb(pool.as_dict()),pool.weights*partial(pool.as_dict()),pool.weights*misid(pool.as_dict())]
sizes=[ns,nc,npart,nm]; keys=[8002,8003,8004,8005]
samples=[weighted_resample(jax.random.key(k),pool,w,n,replace=True) for k,w,n in zip(keys,weights,sizes)]
def merge(ss):
    def cat(name):
        v=[getattr(s,name) for s in ss]; return None if v[0] is None else jnp.concatenate(v)
    return PhaseSpaceSample(s12=cat('s12'),s13=cat('s13'),s23=cat('s23'),weights=jnp.ones(sum(s.size for s in ss)),p1=cat('p1'),p2=cat('p2'),p3=cat('p3'))
data=merge(samples)
print('signal/background:',ns,nb,'background categories:',nc,npart,nm)


## 3. Non-extended fit

P = f_sig S + (1-f_sig)[w_comb B_comb + w_partial B_partial + (1-w_comb-w_partial) B_misID].


In [ ]:
signal_pdf=model.pdf(); d=data.as_dict()
def integ(shape): return jnp.mean(norm.weights*shape(norm.as_dict()))
fs=Parameter('signal_fraction',0.62,bounds=(0.01,0.99),step=0.01)
wc=Parameter('comb_relative_fraction',0.40,bounds=(0.001,0.95),step=0.01)
wp=Parameter('partial_relative_fraction',0.20,bounds=(0.001,0.95),step=0.01)
cats=(BackgroundCategory('comb',comb(d),integ(comb),fraction=wc),BackgroundCategory('partial',partial(d),integ(partial),fraction=wp),BackgroundCategory('misid',misid(d),integ(misid)))
nll=MultiBackgroundNLL(signal_density=lambda v:signal_pdf(d,v),backgrounds=cats,signal_fraction=fs)
pars=(fs,wc,wp); start={'signal_fraction':0.62,'comb_relative_fraction':0.40,'partial_relative_fraction':0.20}
res=Minimizer(nll,pars,verbose=1).fit(start_values=start,simplex=True,ncall=20000)
fit={p.name:float(res.values[p.name]) for p in pars}
print('valid:',res.valid)
print(f"{'parameter':28s} {'generated':>12s} {'start':>12s} {'fitted':>12s}")
for name,truth in [('signal_fraction',fs_true),('comb_relative_fraction',wc_true),('partial_relative_fraction',wp_true)]: print(f"{name:28s} {truth:12.5f} {start[name]:12.5f} {fit[name]:12.5f}")
print('all fitted background weights:',np.asarray(nll.background_weights(fit)))


## 4. Extended form with independent yields


In [ ]:
Nsig=Parameter('signal_yield',25000,bounds=(1,100000),step=100)
Ncomb=Parameter('comb_yield',3000,bounds=(0.1,50000),step=50)
Npartial=Parameter('partial_yield',2000,bounds=(0.1,50000),step=50)
Nmisid=Parameter('misid_yield',1000,bounds=(0.1,50000),step=50)
ext_cats=(BackgroundCategory('comb',comb(d),integ(comb),yield_=Ncomb),BackgroundCategory('partial',partial(d),integ(partial),yield_=Npartial),BackgroundCategory('misid',misid(d),integ(misid),yield_=Nmisid))
ext=MultiBackgroundNLL(signal_density=lambda v:signal_pdf(d,v),backgrounds=ext_cats,extended=True,signal_yield=Nsig)
truth_yields={'signal_yield':ns,'comb_yield':nc,'partial_yield':npart,'misid_yield':nm}
print('expected events at generated yields:',float(ext.expected_events(truth_yields)))
